# Parking Spaces in Berlin 

## Step 1: Research & Data Modelling

**Goals:**

- Discover and document relevant data sources for parking in Berlin.

- Identify key parameters (capacity, fee, operator, geometry).

- Draft a planned target schema for later database population.

- Prepare the `/parking_spaces/sources` directory and store raw data or source links.

### 1.1 Data Source Discovery

See `parking_spaces/sources/README.md`

- OSM OpenStreetMap – public geodata for Berlin 

- Berlin Open Data – Parken im Straßenraum

- Berlin Open Data – Parkraumbewirtschaftung (managed parking zones)

- Berlin Open Data – Park and Ride-Anlagen

- Parkopedia - commercial/licensed, for future enrichment (not in this notebook)

In [6]:
import os
import requests
import xml.etree.ElementTree as ET
import osmnx as ox
import geopandas as gpd
import pandas as pd

from typing import Union
from pathlib import Path
from datetime import datetime

In [12]:
# Prepare sources directory
BASE_DIR = Path.cwd().resolve()
SOURCES_DIR = BASE_DIR.parent / "sources"
SOURCES_DIR.mkdir(parents=True, exist_ok=True)

TODAY = datetime.now().strftime("%Y-%m-%d")
SOURCES_DIR

PosixPath('/Users/didodeboodt/Documents/Projects/Webeet/layered-populate-data-pool-da/parking_spaces/sources')

#### OpenStreetMap (OSM)

**Source and origin:** Public crowdsourced geospatial database

**Update frequency:** Continuous (dynamic)

**Data type:** Dynamic (API query using amenity=parking, amenity=parking_space, amenity=parking_entrance)

**Reason for selection:** Good spatial coverage in Berlin, includes geometry, sometimes capacity and fee, open and free.

**Relevant tags:**

- `amenity=parking` – off-street parking lots  

- `amenity=parking_space` – individual on-street spots  

- `amenity=parking_entrance` – entrances to underground / multi-storey garages  

**Relevant fields:**

- `name`

- `capacity`

- `capacity:disabled`

- `fee`

- `operator`

- `access`

- geometry (lat, lon / polygon)

**Notes:** Good spatial coverage, but attributes like `capacity` or `fee` are not always present. Should be treated as base layer.

**📄 Exported Raw data files to `parking_spaces/sources`:**

- osm_parking_entrances_2025-11-04.geojson

- osm_parking_offstreet_2025-11.04.geojson

- osm_parking_spaces_2025-11-04.geojson


**1. Fetch parking-related data from OpenStreetMap via osmnx:**

In [7]:
TagValue = Union[bool, str, list[str]]

# Off-street style parking: amenity=parkin
tags_parking: dict[str, TagValue] = {"amenity": "parking"}
parking_gdf = ox.features_from_place("Berlin, Germany", tags_parking)
print(f"Off-street parking objects fetched: {len(parking_gdf)}")

# Individual/on-street parking spaces: amenity=parking_space
tags_parking_space: dict[str, TagValue] = {"amenity": "parking_space"}
parking_space_gdf = ox.features_from_place("Berlin, Germany", tags_parking_space)
print(f"Parking space objects fetched: {len(parking_space_gdf)}")

# Parking entrances (garages, underground, etc.)
tags_parking_entrance: dict[str, TagValue] = {"amenity": "parking_entrance"}
parking_entrance_gdf = ox.features_from_place("Berlin, Germany", tags_parking_entrance)
print(f"Parking entrance objects fetched: {len(parking_entrance_gdf)}")

Off-street parking objects fetched: 44159
Parking space objects fetched: 5238
Parking entrance objects fetched: 2472


In [8]:
parking_gdf.head()

geometry     access addr:city  \
element id                                                          
node    29193449   POINT (13.39082 52.51942)  customers    Berlin   
        29697498   POINT (13.36813 52.52729)    private       NaN   
        29707118   POINT (13.35389 52.53832)    private       NaN   
        128349853  POINT (13.27797 52.50831)    private       NaN   
        249500297  POINT (13.17761 52.58533)        NaN       NaN   

                  addr:country addr:housenumber addr:postcode  \
element id                                                      
node    29193449            DE               30         10117   
        29697498           NaN              NaN           NaN   
        29707118           NaN              NaN           NaN   
        128349853          NaN              NaN           NaN   
        249500297          NaN              NaN           NaN   

                       addr:street addr:suburb  amenity  fee  ...  \
element id                                                    ...   
node    29193449   Dorotheenstraße       Mitte  parking  yes  ...   
        29697498               NaN         NaN  parking  NaN  ...   
        29707118               NaN         NaN  parking  NaN  ...   
        128349853              NaN         NaN  parking  NaN  ...   
        249500297              NaN         NaN  parking  NaN  ...   

                  width:street_side cycleway:both maxspeed:type  \
element id                                                        
node    29193449                NaN           NaN           NaN   
        29697498                NaN           NaN           NaN   
        29707118                NaN           NaN           NaN   
        128349853               NaN           NaN           NaN   
        249500297               NaN           NaN           NaN   

                  name:etymology:wikidata sidewalk:left source:maxspeed  \
element id                                                                
node    29193449                      NaN           NaN             NaN   
        29697498                      NaN           NaN             NaN   
        29707118                      NaN           NaN             NaN   
        128349853                     NaN           NaN             NaN   
        249500297                     NaN           NaN             NaN   

                  zone:maxspeed capacity:emergency parking:both:informal  \
element id                                                                 
node    29193449            NaN                NaN                   NaN   
        29697498            NaN                NaN                   NaN   
        29707118            NaN                NaN                   NaN   
        128349853           NaN                NaN                   NaN   
        249500297           NaN                NaN                   NaN   

                  noname  
element id                
node    29193449     NaN  
        29697498     NaN  
        29707118     NaN  
        128349853    NaN  
        249500297    NaN  

[5 rows x 256 columns]

**2. Save each layer to sources directory:**

In [ ]:
offstreet_path = SOURCES_DIR / f"osm_parking_offstreet_{TODAY}.geojson"
parking_gdf.to_file(offstreet_path, driver="GeoJSON")

onstreet_path = SOURCES_DIR / f"osm_parking_spaces_{TODAY}.geojson"
parking_space_gdf.to_file(onstreet_path, driver="GeoJSON")

entrance_path = SOURCES_DIR / f"osm_parking_entrances_{TODAY}.geojson"
parking_entrance_gdf.to_file(entrance_path, driver="GeoJSON")

print("✅ Files saved!")

✅ Files saved!


#### Berlin Open Data – Parken im Straßenraum

**Source and origin:** Berlin Open Data Portal – parking in the street space. Describes all street parking spaces in the state of Berlin. ([WFS](https://gdi.berlin.de/services/wfs/parkplaetze))

**Update frequency:** Marked as updated **26.06.2025** on the portal. (Check portal for later revisions.)  

**Data type:** static (WFS, GeoJSON) 

**Relevant fields:**

- `errechnete_anzahl_parkplaetze`

- `beschraenkungen_variieren_ueber_wochentage`

- `polygon_id`

- `ausrichtung`

- `parkort`

- `parkgebuehr`

- `strassenname`

- `carsharing`

- `nur_schwerbehinderte`

- `bewirtschaftungszeit`

- `zone`

- `oeffentliches_strassenland`

- `bezirk`

- `planungsraum`

- `bezirksregion`

- `coordinates`

**Link:** Berlin Open Data - [Parken im Strassenraum](https://daten.berlin.de/datensaetze/parken-im-strassenraum-wfs-2eb40df3)

**Notes:** Service split into inner/outer ring → both needed for full Berlin coverage.

**📄 Exported Raw data files to `parking_spaces/sources`:** bod_parking_street.geojson (includes parking inside and outside the S-Bahn)


**1. Set WFS URL**

In [ ]:
WFS_URL = (
    "https://gdi.berlin.de/services/wfs/parkplaetze"
    "?service=WFS"
    "&version=2.0.0"
    "&request=GetFeature"
    "&typeNames=fis:parkplaetze_strassenraum"
    "&outputFormat=application/json"
)

**2. Check available services:**

In [14]:
CAP_URL = "https://gdi.berlin.de/services/wfs/parkplaetze?REQUEST=GetCapabilities&SERVICE=WFS"

r = requests.get(CAP_URL)
r.raise_for_status()

root = ET.fromstring(r.content)

ns = {
    "wfs": "http://www.opengis.net/wfs/2.0",
    "ows": "http://www.opengis.net/ows/1.1",
}

layers = []
for ft in root.findall(".//wfs:FeatureType", ns):
    name_el = ft.find("wfs:Name", ns)
    title_el = ft.find("wfs:Title", ns)

    name = name_el.text if name_el is not None else None
    title = title_el.text if title_el is not None else None

    layers.append((name, title))

for name, title in layers:
    print(name, "→", title)

parkplaetze:parkplaetze_aussen → Straßenparkplätze außerhalb des Berliner S-Bahnringes
parkplaetze:parkplaetze → Straßenparkplätze innerhalb des Berliner S-Bahnringes


➡️ There are two available services, parking spaces inside the S-Bahn and parking spaces outside the S-Bahn. We will fetch both and combine them into one geojson file.

**3. Fetch parking data inside Berlin S-Bahn:**

In [15]:
BASE_WFS = "https://gdi.berlin.de/services/wfs/parkplaetze"

inside_url = (
    f"{BASE_WFS}"
    "?service=WFS"
    "&version=2.0.0"
    "&request=GetFeature"
    "&typeNames=parkplaetze:parkplaetze"
    "&outputFormat=application/json"
)

gdf_inside = gpd.read_file(inside_url)
print("inside:", len(gdf_inside))
gdf_inside.head()

inside: 45917


,id,gisid,markierung_parkraum,beschraenkungen_variieren_ueber_wochentage,errechnete_anzahl_parkplaetze,polygon_id,ausrichtung,parkort,parkgebuehr,geltungszeit_der_beschraenkung,...,nur_schwerbehinderte,bewirtschaftungszeit,zone,oeffentliches_strassenland,straßen_id,bezirk,planungsraum,bezirksregion,prognoseraum,geometry
0,parkplaetze.1,1,nicht markiert,nein,3,P262_000000,Längs,Fahrbahn,"1,00 Euro",,...,nein,Mo-Fr 09-20 Uhr / Sa 09-18 Uhr\n,120,Ja,S262_000001,Charlottenburg-Wilmersdorf,Prager Platz,Volkspark Wilmersdorf,Wilmersdorf Zentrum,"MULTIPOLYGON (((386828.038 5817163.179, 386826..."
1,parkplaetze.2,2,nicht markiert,nein,5,P262_000001,Längs,Fahrbahn,"1,00 Euro",,...,nein,Mo-Fr 09-20 Uhr / Sa 09-18 Uhr\n,120,Ja,S262_000001,Charlottenburg-Wilmersdorf,Prager Platz,Volkspark Wilmersdorf,Wilmersdorf Zentrum,"MULTIPOLYGON (((386824.786 5817146.342, 386822..."
2,parkplaetze.3,3,nicht markiert,nein,2,P262_000002,Längs,Fahrbahn,"1,00 Euro",,...,nein,Mo-Fr 09-20 Uhr / Sa 09-18 Uhr\n,120,Ja,S262_000001,Charlottenburg-Wilmersdorf,Prager Platz,Volkspark Wilmersdorf,Wilmersdorf Zentrum,"MULTIPOLYGON (((386824.043 5817115.781, 386822..."
3,parkplaetze.4,4,nicht markiert,nein,6,P262_000003,Längs,Fahrbahn,"1,00 Euro",,...,nein,Mo-Fr 09-20 Uhr / Sa 09-18 Uhr\n,120,Ja,S262_000077,Charlottenburg-Wilmersdorf,Prager Platz,Volkspark Wilmersdorf,Wilmersdorf Zentrum,"MULTIPOLYGON (((386823.536 5817101.67, 386821...."
4,parkplaetze.5,5,nicht markiert,nein,11,P262_000004,Schräg,Fahrbahn,"1,00 Euro",,...,nein,Mo-Fr 09-20 Uhr / Sa 09-18 Uhr\n,120,Ja,S262_000001,Charlottenburg-Wilmersdorf,Prager Platz,Volkspark Wilmersdorf,Wilmersdorf Zentrum,"MULTIPOLYGON (((386824.92 5817063.078, 386819...."


**4. Fetch parking data outside Berlin S-Bahn:**

In [16]:
outside_url = (
    f"{BASE_WFS}"
    "?service=WFS"
    "&version=2.0.0"
    "&request=GetFeature"
    "&typeNames=parkplaetze:parkplaetze_aussen"
    "&outputFormat=application/json"
)

gdf_outside = gpd.read_file(outside_url)
print("outside:", len(gdf_outside))
gdf_outside.head()

outside: 214173


,id,polygonid,ausrichtung,strassenname,zone,anzahl_parkplaetze,strassenid,category,geltungszeit_ladezone,ladezone_einschraenkungen,oeffentlichesstrassenland,bezirk,planungsraum,bezirksregion,prognoseraum,geometry
0,parkplaetze_aussen.B210_000010,B210_000010,Quer,Ottomar-Geschke-Straße,nicht bewirtschaftet,63,S210_000026,Parken (ohne Beschränkungen),,nein,Ja,Treptow-Köpenick,Spindlersfeld,Köllnische Vorstadt/Spindlersfeld,Treptow-Köpenick 2,"MULTIPOLYGON (((402408.993 5811786.149, 402375..."
1,parkplaetze_aussen.B231_000001,B231_000001,Quer,Wartenberger Straße,nicht bewirtschaftet,39,S231_000027,Beschränkungen,,nein,Nein,Lichtenberg,Zingster Straße Ost,Neu-Hohenschönhausen Süd,Hohenschönhausen Nord,"MULTIPOLYGON (((399051.547 5825059.665, 399077..."
2,parkplaetze_aussen.B248_000000,B248_000000,Quer,Waldowstraße,nicht bewirtschaftet,33,S248_000106,Parken (ohne Beschränkungen),,nein,Ja,Lichtenberg,Orankesee,Alt-Hohenschönhausen Süd,Hohenschönhausen Süd,"MULTIPOLYGON (((397870 5822800.502, 397856.376..."
3,parkplaetze_aussen.B258_000004,B258_000004,Quer,Fürstenwalder Damm,nicht bewirtschaftet,72,S258_000282,Parken (ohne Beschränkungen),,nein,Nein,Treptow-Köpenick,Rahnsdorf,Rahnsdorf,Treptow-Köpenick 5,"MULTIPOLYGON (((410155.401 5811319.585, 410119..."
4,parkplaetze_aussen.B288_000010,B288_000010,Quer,Rudolf-Leonhard-Straße,nicht bewirtschaftet,75,S288_000059,Parken (ohne Beschränkungen),,nein,Nein,Marzahn-Hellersdorf,Lea-Grundig-Straße,Marzahn Mitte,Marzahn,"MULTIPOLYGON (((402702.151 5823364.935, 402695..."


**5. Combine data from inside and outside Berlin S-Bahn:**

In [19]:
parking_all = gpd.GeoDataFrame(
    pd.concat([gdf_inside, gdf_outside], ignore_index=True),
    crs=gdf_inside.crs,
)

print("total:", len(parking_all))
parking_all.head()

total: 260090


,id,gisid,markierung_parkraum,beschraenkungen_variieren_ueber_wochentage,errechnete_anzahl_parkplaetze,polygon_id,ausrichtung,parkort,parkgebuehr,geltungszeit_der_beschraenkung,...,bezirksregion,prognoseraum,geometry,polygonid,anzahl_parkplaetze,strassenid,category,geltungszeit_ladezone,ladezone_einschraenkungen,oeffentlichesstrassenland
0,parkplaetze.1,1.0,nicht markiert,nein,3.0,P262_000000,Längs,Fahrbahn,"1,00 Euro",,...,Volkspark Wilmersdorf,Wilmersdorf Zentrum,"MULTIPOLYGON (((386828.038 5817163.179, 386826...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,parkplaetze.2,2.0,nicht markiert,nein,5.0,P262_000001,Längs,Fahrbahn,"1,00 Euro",,...,Volkspark Wilmersdorf,Wilmersdorf Zentrum,"MULTIPOLYGON (((386824.786 5817146.342, 386822...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,parkplaetze.3,3.0,nicht markiert,nein,2.0,P262_000002,Längs,Fahrbahn,"1,00 Euro",,...,Volkspark Wilmersdorf,Wilmersdorf Zentrum,"MULTIPOLYGON (((386824.043 5817115.781, 386822...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,parkplaetze.4,4.0,nicht markiert,nein,6.0,P262_000003,Längs,Fahrbahn,"1,00 Euro",,...,Volkspark Wilmersdorf,Wilmersdorf Zentrum,"MULTIPOLYGON (((386823.536 5817101.67, 386821....",NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,parkplaetze.5,5.0,nicht markiert,nein,11.0,P262_000004,Schräg,Fahrbahn,"1,00 Euro",,...,Volkspark Wilmersdorf,Wilmersdorf Zentrum,"MULTIPOLYGON (((386824.92 5817063.078, 386819....",NaN,NaN,NaN,NaN,NaN,NaN,NaN


**6. Save to sources directory:**

In [20]:
parking_all.to_file(SOURCES_DIR / f"bod_parking_street.geojson", driver="GeoJSON")
print("✅ File saved!")

✅ File saved!


#### Park and Ride-Anlagen

**Sources and origin**: Berlin Open Data - Park and Ride-Anlagen, published by Senatsverwaltung Mobilität, Verkehr, Klimaschutz und Umwelt. Locations and information on Park and Ride (P+R) facilities in the state of Berlin ([WFS](https://gdi.berlin.de/services/wfs/park_and_ride))

**Update frequency**: as published by Senatsverwaltung

**Data Type**: Static WFS (Web Feature Service) providing georeferenced locations of Park & Ride facilities in Berlin.

**Relevant fields**: 

- `bezirk`

- `anlagennam` (Name of the facility)

- `bahnhofsna` (Station name)

- `anzahl_anl` (Count of P+R facilities at station)

- `stellplaet` (Total parking spaces)

- `steplaetze` (Disabled spaces)

- `stellpla_1` (Short-term parking)

- `stellpla_2` (Motorbike spaces)

- `auslastung` (Average occupancy during peak)

- `art` (Type of parking (surface, garage, etc.))

- `belag` (Surface type (asphalt, gravel, etc.))

- `einschraen` (Time restriction (e.g., 24h, 12h))

- `bewirtscha` (Managed/Unmanaged)

- `art_bewirt` (Type of management (free, paid, permit))

- `anzahl_b_u` (Number of bicycle+ride spaces)

**Links**: Berlin Open Data - [Park and Ride-Anlagen](https://daten.berlin.de/datensaetze/park-and-ride-anlagen-wfs-c9d9f2e4)

**Notes:** Enrichment parking layer with transit-oriented parking

**📄 Exported Raw data files to `parking_spaces/sources`:** bod_park_and_ride.geojson

**1. Check available services:**

In [21]:
P_R_CAP_URL = "https://gdi.berlin.de/services/wfs/park_and_ride?REQUEST=GetCapabilities&SERVICE=WFS"

res = requests.get(P_R_CAP_URL)
res.raise_for_status()

root = ET.fromstring(res.content)

ns = {
    "wfs": "http://www.opengis.net/wfs/2.0",
    "ows": "http://www.opengis.net/ows/1.1",
}

layers = []
for ft in root.findall(".//wfs:FeatureType", ns):
    name_el = ft.find("wfs:Name", ns)
    title_el = ft.find("wfs:Title", ns)

    name = name_el.text if name_el is not None else None
    title = title_el.text if title_el is not None else None

    layers.append((name, title))

for name, title in layers:
    print(name, "→", title)

park_and_ride:park_and_ride → Park and Ride-Anlagen


**2. Fetch park and ride data:**

In [22]:
BASE_WFS = "https://gdi.berlin.de/services/wfs/park_and_ride"

typename = "park_and_ride:park_and_ride"  

pandr_url = (
    f"{BASE_WFS}"
    "?service=WFS"
    "&version=2.0.0"
    "&request=GetFeature"
    f"&typeNames={typename}"
    "&outputFormat=application/json"
)

gdf_pandr = gpd.read_file(pandr_url)
print(len(gdf_pandr))
gdf_pandr.head()

48


,id,bezirk,anlagennam,bahnhofsna,anzahl_anl,tarifgebie,r_s_u_lini,stellplaet,steplaetze,stellpla_1,stellpla_2,auslastung,art,belag,einschraen,bewirtscha,art_bewirt,anzahl_b_u,geometry
0,1,Charlottenburg-Wilmersdorf,Breitenbachplatz P1,U Breitenbachplatz,2,B,U3,80,0,0,0,über 90%,ebenerdig; unter Autobahnzubringer,Pflaster,None,nein,None,93,POINT (385086.37 5814426.17)
1,3,Charlottenburg-Wilmersdorf,Heidelberger Platz P1,S+U Heidelberger Platz,1,A,"S41, S42, S46, U3",139,0,0,0,75-90%,ebenerdig; unter Stadtautobahn,Asphalt,None,nein,None,24,POINT (385327.83 5815875.5)
2,4,Charlottenburg-Wilmersdorf,Jungfernheide P1,S+U+R Jungfernheide,2,A,"RE4, RB10, RB21, S41, S42, U7",32,0,0,1,über 90%,ebenerdig,Pflaster,None,nein,None,64,POINT (384705.87 5821440.81)
3,5,Charlottenburg-Wilmersdorf,Jungfernheide P2,S+U+R Jungfernheide,2,A,"RE4, RB10, RB21, S41, S42, U7",67,0,0,0,über 90%,ebenerdig,Pflaster,None,nein,None,64,POINT (384482.94 5821394.17)
4,6,Lichtenberg,Hohenschönhausen P1,S+R Hohenschönhausen,1,B,"RB12, RB24, RB32, S75",47,1,0,0,bis 75%,ebenerdig,Pflaster,None,nein,None,70,POINT (399074.27 5825053.76)


**3. Save to sources directory:**

In [27]:
out_path = SOURCES_DIR / f"bod_park_and_ride.geojson"
gdf_pandr.to_file(out_path, driver="GeoJSON")
print("✅ File saved!")

✅ File saved!


#### Berlin Open Data - Parkraumbewirtschaftung (managed zones)

**Source and origin:** Berlin Open Data – parkraumbewirtschaftungsgebiete (zones where parking is controlled). Shows polygons of zones managed by the Bezirke. ([WFS](https://daten.berlin.de/datensaetze/parkraumbewirtschaftung-wfs-86a217cc))

**Update frequency:** Published by districts; assume occasional changes → treat as periodic static import.  

**Data type:** Static polygon layer.  

**Relevant fields:**

- zone `id` / name

- `Bezirk`

- `Zeiten` 

- `gebuehr`

- `geometry` (polygon)

**Link:** Berlin Open Data - [Parkraumbewirtschaftung](https://daten.berlin.de/datensaetze/parkraumbewirtschaftung-wfs-86a217cc)

**📄 Exported Raw data files to `parking_spaces/sources`:** bod_parking_zones.geojson

**1. Check available services:**

In [24]:
P_R_CAP_URL = "https://gdi.berlin.de/services/wfs/parkraumbewirtschaftung?REQUEST=GetCapabilities&SERVICE=wfs"

res = requests.get(P_R_CAP_URL)
res.raise_for_status()

root = ET.fromstring(res.content)

ns = {
    "wfs": "http://www.opengis.net/wfs/2.0",
    "ows": "http://www.opengis.net/ows/1.1",
}

layers = []
for ft in root.findall(".//wfs:FeatureType", ns):
    name_el = ft.find("wfs:Name", ns)
    title_el = ft.find("wfs:Title", ns)

    name = name_el.text if name_el is not None else None
    title = title_el.text if title_el is not None else None

    layers.append((name, title))

for name, title in layers:
    print(name, "→", title)

parkraumbewirtschaftung:parkzonen → Parkraumbewirtschaftung


**2. Fetch Parkraumbewirtschaftung data:**

In [25]:
BASE_WFS = "https://gdi.berlin.de/services/wfs/parkraumbewirtschaftung"

typename = "parkraumbewirtschaftung:parkzonen"  

pandr_url = (
    f"{BASE_WFS}"
    "?service=WFS"
    "&version=2.0.0"
    "&request=GetFeature"
    f"&typeNames={typename}"
    "&outputFormat=application/json"
)

gdf_pandr = gpd.read_file(pandr_url)
print(len(gdf_pandr))
gdf_pandr.head()

94


,id,parkzone,bezirk,zeiten,gebuehr,bemerkung,geometry
0,parkzonen.1,1,Mitte,Mo-Sa 9-22 Uhr,"4,00 Euro",Gebührenhöhe und Bewirtschaftungszeiten können...,"MULTIPOLYGON (((390868.524 5820388.19, 390868...."
1,parkzonen.10,10,Spandau,"Mo-Fr 9-17 Uhr, Sa 9 -14 Uhr/ Advents-Sa 9 -17...","2,00 Euro",Gebührenhöhe und Bewirtschaftungszeiten können...,"MULTIPOLYGON (((377895.364 5822002.656, 377965..."
2,parkzonen.100,100,Neukölln,Mo-Fr 9-20 Uhr\n,"4,00 Euro",Gebührenhöhe und Bewirtschaftungszeiten können...,"MULTIPOLYGON (((394070.04 5815727.048, 394084...."
3,parkzonen.101,101,Neukölln,Mo-Fr 9-20 Uhr,"4,00 Euro",Gebührenhöhe und Bewirtschaftungszeiten können...,"MULTIPOLYGON (((393754.929 5815654.204, 393638..."
4,parkzonen.105,105,Neukölln,Mo-Fr 9-20 Uhr\n,"4,00 Euro",Gebührenhöhe und Bewirtschaftungszeiten können...,"MULTIPOLYGON (((393093.315 5816495.726, 393066..."


**3. Save to source directory:**

In [28]:
out_path = SOURCES_DIR / f"bod_parking_zones.geojson"
gdf_pandr.to_file(out_path, driver="GeoJSON")

print("✅ File saved!")

✅ File saved!


### 1.2 Modelling & Planning

#### Parking Table Schema Draft

This is a **proposal** for how to map all above sources into a single raw-ish table later:

| **field name**           | **description**                                              |
|--------------------------|----------------------------------------------------------|
| source                   | `osm`, `bod_parken`, `bod_parkandride`, …                |
| source_layer             | sublayer or OSM tag combination                          |
| external_id              | id from source (OSM id, WFS id, …)                       |
| name                     | parking name or description                              |
| parking_type             | e.g. `off_street`, `on_street`, `garage`, `zone`, `P&R`  |
| operator                 | city / private / district                                |
| fee                      | boolean or string from source                            |
| time restriction         | 12h, 24h, ..                                             |
| capacity                 | integer                                                  |
| capacity_disabled        | integer if available                                     |
| street_name              | from WFS if present                                      |
| district                 | Berlin Bezirk if                                         |
| managed_zone_id          | id from parkraumbewirtschaftung                          |
| geometry_type            | point / polygon / line                                   |
| geometry                 | geom                                                     |
| last_updated_at_source   | date from feed if present                                |
| fetched_at               | timestamp of download                                    |

#### Planned Data Transformation Steps

- Normalize column names (lowercase, snakecase, remove special characters)

- Select relevant columns

- Drop columns with >85% null

- Remove duplicates (deduplicate on same geometry centroid within 5–10 meters and same name)

- Remove empty geometries, fix invalid geometries, explode multiparts

- Map source-specific categories to common parking_type

  - OSM amenity=parking_space → on_street

  - WFS street parking → on_street

  - WFS P+R → park_and_ride

  - WFS parkraumbewirtschaftung → zone
  
  - OSM amenity=parking with parking=multi-storey → garage

#### Data issues or inconsistencies

**Different field names and languages**: Berlin Open Data fields are in German (bezirk, strassenname, stellplaet), while OSM uses English and underscores (capacity, fee, operator). → Requires normalization to a unified schema.

**Data types inconsistencies**: Numeric fields like capacity or fee can appear as strings in WFS responses ("30" instead of 30, "ja"/"nein" for booleans).

**Missing IDs**: Some WFS datasets have no stable unique identifier and mostly have ploygon_id, while OSM has stable unique identifiers.

**Uneven spatial coverages**

- OSM data coverage depends on community mapping (some districts may be better mapped than others)
- Park & Ride only covers specific sites and not all large parking spaces
- Parken im Straßenraum does not include private or informal street parking

**Split services**: Parken im Straßenraum is split into inner and outer S-Bahn ring WFS layers. When combining them they can cause duplicates or missing areas if one of the layers fails.

**Misaligned Geometry**: Some WFS geometries may not perfectly align with official district boundaries or OSM geometries (could give issues with spatial joins)

**Missing values**: Some columns might be outdated or have a lot of missing data

**Unverified values**: OSM attributes are entered by users and may contain typos or inconsistencies 

**Overlapping sources**:

- OSM and Parken im Straßenraum may both describe the same locations differently
- Park & Ride locations can also appear as amenity=parking in OSM

**Coordinate Reference System (CRS)**: There are inconcistent CRS across sources (Some WFS services deliver data in EPSG:25833 (ETRS89 / UTM Zone 33N), others in EPSG:4326 (WGS84).) 

**License & Accessibility**

- OSM data is under ODbL license (requires attribution)
- Berlin Open Data typically uses DL-DE Zero 2.0
- Parkopedia is commercial and cannot be redistributed without permission


### 1.3 Prepare the `sources` directory

All extracted raw data files have been added to `parking_spaces/sources` and a detailed `README.md` has been added to the directory as well. 